## 1. Import and Load

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
df_raw = pd.read_csv('/Users/mac/Desktop/5205 Andrew/proposal/Cleaned_data_Spotify_weekly_top200_2017-2025.csv')

In [7]:
df_raw.head()

,Week_End,Week_Start_Fri,Week_Label,Weekly_Rank,id,Title,Artists_All,Weekly_Points,Streams,Days_Charted,...,Stage,Year,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,Valence,Song_URL
0,2017/1/6,2016/12/31,2016-12-31 → 2017-01-06,1,5aAx2yezTd8zXrkmtKl66Z,Starboy,"The Weeknd, Daft Punk",1198,NaN,6.0,...,Stage 1 (2017-2019),2017,0.681,0.594,-7028.0,0.282,0.165,0.0,0.535,https://open.spotify.com/track/5aAx2yezTd8zXrk...
1,2017/1/6,2016/12/31,2016-12-31 → 2017-01-06,2,7BKLCZ1jbUBVqRi2FVlTVw,Closer,"The Chainsmokers, Halsey",1192,NaN,6.0,...,Stage 1 (2017-2019),2017,0.748,0.524,-5599.0,0.034,0.414,0.0,0.661,https://open.spotify.com/track/7BKLCZ1jbUBVqRi...
2,2017/1/6,2016/12/31,2016-12-31 → 2017-01-06,3,5knuzwU65gJK7IF5yJsuaW,Rockabye (feat. Sean Paul & Anne-Marie),Clean Bandit,1185,NaN,6.0,...,Stage 1 (2017-2019),2017,0.720,0.763,-4068.0,0.052,0.406,0.0,0.742,https://open.spotify.com/track/5knuzwU65gJK7IF...
3,2017/1/6,2016/12/31,2016-12-31 → 2017-01-06,4,4pdPtRcBmOSQDlJ3Fk945m,Let Me Love You,"DJ Snake, Justin Bieber",1176,NaN,6.0,...,Stage 1 (2017-2019),2017,0.476,0.718,-5309.0,0.058,0.078,0.0,0.142,https://open.spotify.com/track/4pdPtRcBmOSQDlJ...
4,2017/1/6,2016/12/31,2016-12-31 → 2017-01-06,5,3NdDpSvN911VPGivFlV5d0,I Don't Wan Live Forever (Fifty Shades Darker)...,"ZAYN, Taylor Swift",1170,NaN,6.0,...,Stage 1 (2017-2019),2017,0.735,0.451,-8374.0,0.059,0.063,0.0,0.086,https://open.spotify.com/track/3NdDpSvN911VPGi...


In [8]:
print(df_raw[['Danceability', 'Energy', 'Loudness', 'Speechiness', 
          'Acousticness', 'Instrumentalness', 'Valence']].isnull().sum())

Danceability        2329
Energy              2329
Loudness            2329
Speechiness         2329
Acousticness        2329
Instrumentalness    2329
Valence             2329
dtype: int64


In [9]:
print(f"Shape: {df_raw.shape}")

Shape: (94052, 21)


In [21]:
print(f"\nStage distribution:\n{df_raw['Stage'].value_counts()}")


Stage distribution:
Stage
Stage 2 (2020-2022)    31429
Stage 3 (2023-2025)    31401
Stage 1 (2017-2019)    31222
Name: count, dtype: int64


## 2.Fix Loudness: Loudness should range in (-60, 0)

In [10]:
df = df_raw.copy()

In [11]:
print(df['Loudness'].describe())
print(df['Loudness'].head(20))

count    91723.000000
mean     -4776.473110
std       3346.966165
min     -25166.000000
25%      -6862.000000
50%      -5044.000000
75%      -2824.000000
max       1509.000000
Name: Loudness, dtype: float64
0    -7028.00
1    -5599.00
2    -4068.00
3    -5309.00
4    -8374.00
5    -6126.00
6       -9.35
7    -7398.00
8    -5946.00
9    -4282.00
10   -6163.00
11   -5886.00
12   -2921.00
13   -4023.00
14   -6237.00
15   -5092.00
16   -5314.00
17   -3515.00
18   -5883.00
19   -4757.00
Name: Loudness, dtype: float64


In [12]:
# Fix Loudness values that were multiplied by 1000
df['Loudness'] = df['Loudness'].apply(lambda x: x / 1000 if x < -100 else x)
print(df['Loudness'].describe())
print(df['Loudness'].head(20))

count    91723.000000
mean        -6.227580
std         10.402675
min        -54.341000
25%         -7.395000
50%         -5.795000
75%         -4.554000
max       1509.000000
Name: Loudness, dtype: float64
0    -7.028
1    -5.599
2    -4.068
3    -5.309
4    -8.374
5    -6.126
6    -9.350
7    -7.398
8    -5.946
9    -4.282
10   -6.163
11   -5.886
12   -2.921
13   -4.023
14   -6.237
15   -5.092
16   -5.314
17   -3.515
18   -5.883
19   -4.757
Name: Loudness, dtype: float64


In [14]:
# Remove Loudness outliers outside normal range (-60 to 0)
print(f"Rows with Loudness > 0: {(df['Loudness'] > 0).sum()}")
print(f"Rows with Loudness < -60: {(df['Loudness'] < -60).sum()}")

Rows with Loudness > 0: 14
Rows with Loudness < -60: 0


In [15]:
df = df[(df['Loudness'] >= -60) & (df['Loudness'] <= 0)].reset_index(drop=True)

In [16]:
print(f"Shape after Loudness fix: {df.shape}")
print(f"\nLoudness stats after fix:\n{df['Loudness'].describe()}")

Shape after Loudness fix: (91709, 21)

Loudness stats after fix:
count    91709.000000
mean        -6.294381
std          2.843034
min        -54.341000
25%         -7.395000
50%         -5.795000
75%         -4.554000
max         -0.484000
Name: Loudness, dtype: float64


## 3. Handle Missing Values

In [17]:
FEATURES = ['Danceability', 'Energy', 'Loudness', 'Speechiness', 
            'Acousticness', 'Instrumentalness', 'Valence']

In [18]:
df_clean = df.dropna(subset=FEATURES).reset_index(drop=True) 
print(f"Shape: {df_clean.shape}")

Shape: (91709, 21)


In [19]:
print(df_clean[FEATURES].isnull().sum())

Danceability        0
Energy              0
Loudness            0
Speechiness         0
Acousticness        0
Instrumentalness    0
Valence             0
dtype: int64


In [20]:
print(f"\nStage distribution:\n{df_clean['Stage'].value_counts()}")


Stage distribution:
Stage
Stage 2 (2020-2022)    31425
Stage 1 (2017-2019)    31217
Stage 3 (2023-2025)    29067
Name: count, dtype: int64


In [ ]:
df_clean.to_csv('/Users/mac/ML-spotify-features/Spotify_weekly_top200_cleaned.csv', index=False)